# ML_training — Prévision TempRet / PuisCpt par EGID

Exploration **multi-modèles** (Random Forest, XGBoost, LSTM) sur des séries au pas 15 min, par cluster.

**Prérequis :** ouvrir ce notebook avec le répertoire de travail = dossier `2_Program/` (chemins relatifs `0_Data/...`).

**Données :** `3_Training`, `4_Validation`, `5_Test` — fichiers `cluster{N}.parquet` (N ∈ {3,4,5,6}).

**Sorties :** modèles dans `0_Data/6_Models/Cluster{N}/` ; prédictions test dans `0_Data/9_Results/Cluster{N}/`.

Principes : split temporel déjà appliqué dans le pipeline ; **pas de fuite** (lags décalés, scaler fit sur train uniquement) ; **RAM** : chargement par EGID, purge après chaque modèle.

**Entrées / cibles** : uniquement des grandeurs **normalisées [0, 1]** comme en `dataset_preparation_V2` (section 6) : exogènes `TempExt_norm` (pas `TempExt`), cibles `TempRet_norm` et `PuisCpt_fc`. Les exports et graphiques **repassent `TempRet` en °C** (inverse de la norme) ; **PuisCpt** reste en facteur de charge fc.


## 1. Configuration

In [1]:
from __future__ import annotations

import gc
import logging
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("ml_training")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# Chemins : cwd = 2_Program
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PATH_TRAIN = NOTEBOOK_DIR / "0_Data" / "3_Training"
PATH_VAL = NOTEBOOK_DIR / "0_Data" / "4_Validation"
PATH_TEST = NOTEBOOK_DIR / "0_Data" / "5_Test"
PATH_MODELS = NOTEBOOK_DIR / "0_Data" / "6_Models"
PATH_RESULTS = NOTEBOOK_DIR / "0_Data" / "9_Results"

CLUSTER_IDS = [3, 4, 5, 6]
SEED = 42
N_EGID_PER_CLUSTER = 5

# Parallélisme RF + XGB (joblib / OpenMP) : n_jobs négatif → max(1, n_cpu + 1 + n_jobs)
N_JOBS_PARALLEL = -5

# Features dérivées (lags / rolling) — aligné skill time-series
LAGS = [1, 2, 3, 7]
ROLL_WINDOWS = [3, 7]

EXOG_COLS = [
    "TempExt_norm",
    "dayofyear_cos",
    "dayofyear_sin",
    "dayofweek_cos",
    "dayofweek_sin",
    "hour_cos",
    "hour_sin",
]

# Random Forest — grille réduite, sélection sur validation
RF_PARAM_GRID = [
    {"n_estimators": 80, "max_depth": 8},
    {"n_estimators": 120, "max_depth": 10},
    {"n_estimators": 150, "max_depth": 12},
    {"n_estimators": 120, "max_depth": None},
]

# XGBoost
XGB_PARAMS = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=N_JOBS_PARALLEL,
    verbosity=0,
)
XGB_EARLY_STOPPING_ROUNDS = 40

# LSTM
SEQ_LEN = 24
LSTM_EPOCHS = 80
LSTM_BATCH_SIZE = 64
LSTM_PATIENCE = 12
LSTM_ADAM_LEARNING_RATE = 1e-3  # optimiseur Adam (compile LSTM)
LSTM_COMPILE_LOSS = "mse"

# Dénormalisation TempRet (°C) — aligné pipeline : clip((T-20)/60, 0, 1)
TEMPRET_NORM_T0_C = 20.0
TEMPRET_NORM_SCALE_C = 60.0

RNG = np.random.default_rng(SEED)

# Graphiques section analyse
PLOT_START = None  # ex. pd.Timestamp("2025-01-01", tz="UTC") ou None = auto (derniers points)
PLOT_END = None
PLOT_MAX_POINTS = 500  # si pas de plage explicite : tracer les N derniers pas temporels valides

SELECTED_EGIDS: dict[int, list[str]] = {}


In [2]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)


def parse_egids(columns: list[str]) -> list[str]:
    """EGID à partir des colonnes *.TempRet_norm."""
    out: set[str] = set()
    for c in columns:
        if c.endswith(".TempRet_norm"):
            out.add(c[: -len(".TempRet_norm")])
    return sorted(out)


def columns_for_egid(egid: str) -> list[str]:
    eg = str(egid)
    return (
        ["Dates"]
        + EXOG_COLS
        + [
            f"{eg}.TempRet_norm",
            f"{eg}.PuisCpt_fc",
            f"{eg}.TempRet.inv",
            f"{eg}.PuisCpt.inv",
        ]
    )


def feature_column_names() -> list[str]:
    feats = list(EXOG_COLS)
    for lag in LAGS:
        feats += [f"lag_tr_{lag}", f"lag_pc_{lag}"]
    for w in ROLL_WINDOWS:
        feats += [
            f"roll_tr_mean_{w}",
            f"roll_tr_std_{w}",
            f"roll_pc_mean_{w}",
            f"roll_pc_std_{w}",
        ]
    return feats


FEATURE_COLS = feature_column_names()


def add_lag_features(df: pd.DataFrame, egid: str) -> pd.DataFrame:
    eg = str(egid)
    tr, pc = f"{eg}.TempRet_norm", f"{eg}.PuisCpt_fc"
    out = df.copy()
    for lag in LAGS:
        out[f"lag_tr_{lag}"] = out[tr].shift(lag)
        out[f"lag_pc_{lag}"] = out[pc].shift(lag)
    shifted_tr = out[tr].shift(1)
    shifted_pc = out[pc].shift(1)
    for w in ROLL_WINDOWS:
        out[f"roll_tr_mean_{w}"] = shifted_tr.rolling(w).mean()
        out[f"roll_tr_std_{w}"] = shifted_tr.rolling(w).std()
        out[f"roll_pc_mean_{w}"] = shifted_pc.rolling(w).mean()
        out[f"roll_pc_std_{w}"] = shifted_pc.rolling(w).std()
    return out


def free_ram(*objs) -> None:
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()


def free_tf() -> None:
    try:
        tf.keras.backend.clear_session()
    except Exception as e:
        logger.warning("clear_session: %s", e)
    gc.collect()


def load_concat_frames(cluster_id: int, egid: str) -> tuple[pd.DataFrame, tuple[int, int, int]]:
    """Charge train/val/test (colonnes minimales), concatène avec marqueur _sp."""
    cols = columns_for_egid(egid)
    tr_path = PATH_TRAIN / f"cluster{cluster_id}.parquet"
    va_path = PATH_VAL / f"cluster{cluster_id}.parquet"
    te_path = PATH_TEST / f"cluster{cluster_id}.parquet"
    dtr = pd.read_parquet(tr_path, columns=cols)
    dva = pd.read_parquet(va_path, columns=cols)
    dte = pd.read_parquet(te_path, columns=cols)
    n_tr, n_va, n_te = len(dtr), len(dva), len(dte)
    dtr = dtr.copy()
    dva = dva.copy()
    dte = dte.copy()
    dtr["_sp"] = 0
    dva["_sp"] = 1
    dte["_sp"] = 2
    full = pd.concat([dtr, dva, dte], ignore_index=True)
    full = add_lag_features(full, egid)
    free_ram(dtr, dva, dte)
    return full, (n_tr, n_va, n_te)


def build_xy_matrices(full: pd.DataFrame, egid: str):
    """Retourne X_raw, y ([TempRet_norm, PuisCpt_fc] dans [0,1]), sp, inv_ok, dates."""
    eg = str(egid)
    tr_c = f"{eg}.TempRet_norm"
    pc_c = f"{eg}.PuisCpt_fc"
    vi_tr = f"{eg}.TempRet.inv"
    vi_pc = f"{eg}.PuisCpt.inv"

    feat_ok = full[FEATURE_COLS].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
    y_ok = full[[tr_c, pc_c]].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
    ready = feat_ok & y_ok

    sub = full.loc[ready].reset_index(drop=True)
    X_raw = sub[FEATURE_COLS].to_numpy(dtype=np.float64)
    y = np.clip(sub[[tr_c, pc_c]].to_numpy(dtype=np.float64), 0.0, 1.0)
    sp = sub["_sp"].to_numpy(dtype=np.int8)
    dates = sub["Dates"].to_numpy()
    inv_ok = (sub[vi_tr].to_numpy() == 0) & (sub[vi_pc].to_numpy() == 0)

    return X_raw, y, sp, inv_ok, dates, sub[[vi_tr, vi_pc]]


def rmse_per_target(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))


def aggregate_score(rmses: np.ndarray) -> float:
    """Combine les deux cibles (moyenne des RMSE) pour comparaison unique."""
    return float(np.mean(rmses))


def ensure_dirs(cluster_id: int) -> tuple[Path, Path]:
    mdir = PATH_MODELS / f"Cluster{cluster_id}"
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    mdir.mkdir(parents=True, exist_ok=True)
    rdir.mkdir(parents=True, exist_ok=True)
    return mdir, rdir


def denorm_tempret_celsius(norm: np.ndarray) -> np.ndarray:
    """Inverse norme pipeline : T(°C) = TEMPRET_NORM_T0_C + scale * clip(norm, 0, 1)."""
    x = np.clip(np.asarray(norm, dtype=np.float64), 0.0, 1.0)
    return TEMPRET_NORM_T0_C + TEMPRET_NORM_SCALE_C * x


def export_test_predictions(
    path_parquet: Path,
    dates: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    inv_df: pd.DataFrame,
) -> None:
    yt = np.clip(np.asarray(y_true, dtype=np.float64), 0.0, 1.0)
    yp = np.clip(np.asarray(y_pred, dtype=np.float64), 0.0, 1.0)
    out = pd.DataFrame(
        {
            "Dates": dates,
            "TempRet": denorm_tempret_celsius(yt[:, 0]),
            "PuisCpt": yt[:, 1],
            "TempRetPred": denorm_tempret_celsius(yp[:, 0]),
            "PuisCptPred": yp[:, 1],
        }
    )
    if inv_df is not None and len(inv_df) == len(out):
        out["TempRet.inv"] = inv_df.iloc[:, 0].to_numpy()
        out["PuisCpt.inv"] = inv_df.iloc[:, 1].to_numpy()
    out.to_parquet(path_parquet, index=False)


def collect_lstm_end_indices(sp: np.ndarray, seq_len: int) -> tuple[list[int], list[int], list[int]]:
    """Indices de fin de fenêtre (inclus) pour train / val / test sans chevauchement de split."""
    n = len(sp)
    train_e, val_e, test_e = [], [], []
    for i in range(seq_len - 1, n):
        start = i - seq_len + 1
        seg = sp[start : i + 1]
        if seg[-1] == 0 and bool(np.all(seg == 0)):
            train_e.append(i)
        elif seg[-1] == 1 and int(seg.max()) <= 1:
            val_e.append(i)
        elif seg[-1] == 2:
            test_e.append(i)
    return train_e, val_e, test_e


def ends_to_seq_X(X: np.ndarray, ends: list[int], seq_len: int) -> np.ndarray:
    return np.stack([X[t - seq_len + 1 : t + 1] for t in ends], axis=0)


In [3]:
def sample_egids_all_clusters() -> dict[int, list[str]]:
    selected: dict[int, list[str]] = {}
    for cid in CLUSTER_IDS:
        cols = pq.ParquetFile(PATH_TRAIN / f"cluster{cid}.parquet").schema.names
        all_e = parse_egids(cols)
        if len(all_e) < N_EGID_PER_CLUSTER:
            raise ValueError(f"Cluster {cid}: seulement {len(all_e)} EGID, besoin de {N_EGID_PER_CLUSTER}")
        pick = RNG.choice(np.array(all_e, dtype=object), size=N_EGID_PER_CLUSTER, replace=False)
        selected[cid] = sorted(str(x) for x in pick.tolist())
    return selected


SELECTED_EGIDS = sample_egids_all_clusters()
for k, v in SELECTED_EGIDS.items():
    print(f"Cluster {k}: {v}")


Cluster 3: ['1511188', '190207608', '190853709', '191670265', '502172326']
Cluster 4: ['1511077', '190607088', '191857383', '191955715', '9081114']
Cluster 5: ['1511098', '1517454', '190216197', '190653848', '3091287']
Cluster 6: ['1511261', '1511810', '160001213', '160001283', '3091382']


## 2. Random Forest — un bloc par cluster (5 EGID, grille HP sur validation, refit train)

In [4]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_te = rf_final.predict(X_te)

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO RF cluster 3 EGID 1511188
INFO RF cluster 3 EGID 190207608
INFO RF cluster 3 EGID 190853709
INFO RF cluster 3 EGID 191670265
INFO RF cluster 3 EGID 502172326
INFO RF terminé cluster 3


In [5]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_te = rf_final.predict(X_te)

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO RF cluster 4 EGID 1511077
INFO RF cluster 4 EGID 190607088
INFO RF cluster 4 EGID 191857383
INFO RF cluster 4 EGID 191955715
INFO RF cluster 4 EGID 9081114
INFO RF terminé cluster 4


In [6]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_te = rf_final.predict(X_te)

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO RF cluster 5 EGID 1511098
INFO RF cluster 5 EGID 1517454
INFO RF cluster 5 EGID 190216197
INFO RF cluster 5 EGID 190653848
INFO RF cluster 5 EGID 3091287
INFO RF terminé cluster 5


In [7]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_te = rf_final.predict(X_te)

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO RF cluster 6 EGID 1511261
INFO RF cluster 6 EGID 1511810
INFO RF cluster 6 EGID 160001213
INFO RF cluster 6 EGID 160001283
INFO RF cluster 6 EGID 3091382
INFO RF terminé cluster 6


## 3. XGBoost — un bloc par cluster (early stopping sur validation)

In [8]:
import xgboost as xgb


In [9]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO XGB cluster 3 EGID 1511188
INFO XGB cluster 3 EGID 190207608
INFO XGB cluster 3 EGID 190853709
INFO XGB cluster 3 EGID 191670265
INFO XGB cluster 3 EGID 502172326
INFO XGB terminé cluster 3


In [10]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO XGB cluster 4 EGID 1511077
INFO XGB cluster 4 EGID 190607088
INFO XGB cluster 4 EGID 191857383
INFO XGB cluster 4 EGID 191955715
INFO XGB cluster 4 EGID 9081114
INFO XGB terminé cluster 4


In [11]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO XGB cluster 5 EGID 1511098
INFO XGB cluster 5 EGID 1517454
INFO XGB cluster 5 EGID 190216197
INFO XGB cluster 5 EGID 190653848
INFO XGB cluster 5 EGID 3091287
INFO XGB terminé cluster 5


In [12]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO XGB cluster 6 EGID 1511261
INFO XGB cluster 6 EGID 1511810
INFO XGB cluster 6 EGID 160001213
INFO XGB cluster 6 EGID 160001283
INFO XGB cluster 6 EGID 3091382
INFO XGB terminé cluster 6


## 4. LSTM — un bloc par cluster (early stopping Keras)

In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout


def build_lstm_model(seq_len: int, n_features: int):
    """Architecture LSTM multi-sortie : Dense(2, sigmoid) = [TempRet_norm, PuisCpt_fc] dans [0,1].
    Modifier **uniquement** ce corps pour les 4 blocs cluster."""
    return Sequential(
        [
            LSTM(64, return_sequences=True, input_shape=(seq_len, n_features)),
            Dropout(0.2),
            LSTM(32, return_sequences=False),
            Dropout(0.2),
            Dense(2, activation="sigmoid"),
        ]
    )


def compile_lstm_model(model) -> None:
    """Compile avec optimiseur Adam explicite (taux dans la cellule Configuration)."""
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LSTM_ADAM_LEARNING_RATE),
        loss=LSTM_COMPILE_LOSS,
        metrics=["mae"],
    )


In [14]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO LSTM cluster 3 EGID 1511188
WARNING From c:\LUA\CASIA\TCas\2_Program\.venv\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

INFO LSTM cluster 3 EGID 190207608
INFO LSTM cluster 3 EGID 190853709
INFO LSTM cluster 3 EGID 191670265
INFO LSTM cluster 3 EGID 502172326
INFO LSTM terminé cluster 3


In [15]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO LSTM cluster 4 EGID 1511077
INFO LSTM cluster 4 EGID 190607088
INFO LSTM cluster 4 EGID 191857383
INFO LSTM cluster 4 EGID 191955715
INFO LSTM cluster 4 EGID 9081114
INFO LSTM terminé cluster 4


In [16]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO LSTM cluster 5 EGID 1511098
INFO LSTM cluster 5 EGID 1517454
INFO LSTM cluster 5 EGID 190216197
INFO LSTM cluster 5 EGID 190653848
INFO LSTM cluster 5 EGID 3091287
INFO LSTM terminé cluster 5


In [17]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


INFO LSTM cluster 6 EGID 1511261
INFO LSTM cluster 6 EGID 1511810
INFO LSTM cluster 6 EGID 160001213
INFO LSTM cluster 6 EGID 160001283
INFO LSTM cluster 6 EGID 3091382
INFO LSTM terminé cluster 6


## 5. Analyse et comparaison

- Métriques sur les fichiers test exportés (`RF_*`, `XB_*`, `LSTM_*`).
- Meilleur modèle par EGID (score = moyenne des RMSE sur **TempRet en °C** (déjà dénormé dans les parquet) et **PuisCpt en fc** [0,1]).
- Synthèse par cluster et globale ; graphiques comparatifs (plage configurable).


In [20]:
PREFIXES = ("RF", "XB", "LSTM")


def load_result_metrics(cluster_id: int) -> pd.DataFrame:
    rows = []
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    for egid in SELECTED_EGIDS[cluster_id]:
        for pref in PREFIXES:
            p = rdir / f"{pref}_{egid}.parquet"
            if not p.exists():
                logger.warning("Manquant: %s", p)
                continue
            df = pd.read_parquet(p)
            yt = df[["TempRet", "PuisCpt"]].to_numpy()
            yp = df[["TempRetPred", "PuisCptPred"]].to_numpy()
            mae_t = mean_absolute_error(yt[:, 0], yp[:, 0])
            mae_p = mean_absolute_error(yt[:, 1], yp[:, 1])
            rmse_t = np.sqrt(mean_squared_error(yt[:, 0], yp[:, 0]))
            rmse_p = np.sqrt(mean_squared_error(yt[:, 1], yp[:, 1]))
            score = float(np.mean([rmse_t, rmse_p]))
            rows.append(
                dict(
                    cluster_id=cluster_id,
                    egid=egid,
                    model=pref,
                    rmse_TempRet=rmse_t,
                    rmse_PuisCpt=rmse_p,
                    mae_TempRet=mae_t,
                    mae_PuisCpt=mae_p,
                    score=score,
                )
            )
    return pd.DataFrame(rows)


metrics_all = pd.concat([load_result_metrics(c) for c in CLUSTER_IDS], ignore_index=True)
metrics_all.to_csv(PATH_RESULTS / "metrics_all_models.csv", index=False)
print(metrics_all.groupby("model")["score"].mean())

best_per_egid = metrics_all.sort_values("score").groupby(["cluster_id", "egid"], as_index=False).first()
best_per_egid.to_csv(PATH_RESULTS / "best_model_per_egid.csv", index=False)
print("Meilleur modèle par EGID (extrait):")
print(best_per_egid.head(20))

best_by_cluster = best_per_egid.groupby("cluster_id")["model"].agg(lambda x: x.mode().iloc[0] if len(x) else None)
print("Modèle dominant par cluster (mode des gagnants EGID):")
print(best_by_cluster)

global_counts = best_per_egid["model"].value_counts()
print("Répartition globale des meilleurs modèles par EGID:")
print(global_counts)
best_global = global_counts.index[0] if len(global_counts) else None
print("Meilleur type global (vote EGID):", best_global)


model
LSTM    1.473808
RF      1.501723
XB      1.361991
Name: score, dtype: float64
Meilleur modèle par EGID (extrait):
    cluster_id       egid model  rmse_TempRet  rmse_PuisCpt  mae_TempRet  \
0            3    1511188    XB      0.420859      0.022471     0.203912   
1            3  190207608    XB      3.570024      0.106747     2.148466   
2            3  190853709    XB      5.944630      0.138649     5.247342   
3            3  191670265    XB      4.014410      0.178838     2.556141   
4            3  502172326    XB      0.076825      0.029358     0.053523   
5            4    1511077    XB      1.019172      0.057762     0.757315   
6            4  190607088    RF      0.782789      0.050650     0.581211   
7            4  191857383    XB      2.663390      0.075953     1.613561   
8            4  191955715    XB      3.430169      0.037278     1.915703   
9            4    9081114    XB      0.765676      0.024964     0.492521   
10           5    1511098    XB      1.0246

In [19]:
import matplotlib.pyplot as plt


def plot_egid_comparison(cluster_id: int, egid: str) -> None:
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for pref, style in [("RF", "-"), ("XB", "--"), ("LSTM", ":")]:
        p = rdir / f"{pref}_{egid}.parquet"
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        d = df.copy()
        d["Dates"] = pd.to_datetime(d["Dates"], utc=True)
        if PLOT_START is not None:
            d = d[d["Dates"] >= pd.Timestamp(PLOT_START)]
        if PLOT_END is not None:
            d = d[d["Dates"] <= pd.Timestamp(PLOT_END)]
        if PLOT_START is None and PLOT_END is None and PLOT_MAX_POINTS and len(d) > PLOT_MAX_POINTS:
            d = d.iloc[-PLOT_MAX_POINTS :]
        ax0, ax1 = axes[0], axes[1]
        ax0.plot(d["Dates"], d["TempRet"], color="k", alpha=0.35, linewidth=1, label="_nolegend_" if pref != "RF" else "cible")
        ax0.plot(d["Dates"], d["TempRetPred"], linestyle=style, linewidth=1.2, label=f"{pref} pred")
        ax1.plot(d["Dates"], d["PuisCpt"], color="k", alpha=0.35, linewidth=1, label="_nolegend_" if pref != "RF" else "cible")
        ax1.plot(d["Dates"], d["PuisCptPred"], linestyle=style, linewidth=1.2, label=f"{pref} pred")
    axes[0].set_ylabel("TempRet (°C)")
    axes[1].set_ylabel("PuisCpt (fc)")
    axes[0].set_title(f"Cluster {cluster_id} — EGID {egid}")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[1].legend(loc="upper right", fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    out = rdir / f"compare_{egid}_TempRet_PuisCpt.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Graphique %s", out)


for cid in CLUSTER_IDS:
    for egid in SELECTED_EGIDS[cid]:
        plot_egid_comparison(cid, egid)


INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster3\compare_1511188_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster3\compare_190207608_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster3\compare_190853709_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster3\compare_191670265_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster3\compare_502172326_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster4\compare_1511077_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster4\compare_190607088_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster4\compare_191857383_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Program\0_Data\9_Results\Cluster4\compare_191955715_TempRet_PuisCpt.png
INFO Graphique c:\LUA\CASIA\TCas\2_Progra